In [1]:
import numpy as np
import matplotlib.pyplot as plt
import labmate

In [16]:
import os
import time
import struct
# from utils import write

In [3]:
from labmate.acquisition_notebook import AcquisitionAnalysisManager

DATA_DIR = "tmp_data/"
# aqm = AcquisitionAnalysisManager("DATA_DIR")
# this would save data inside /path/to/current/file/tmp_data/

In [4]:
SCOPE_DEV = "/dev/usbtmc0"

# TEKTRONIK

Simple aquisition of all the displayed traces of the tektronik using serial communication. The data is saved in the `tmp_data` folder, and can be loaded using the `analysis.ipynb` notebook.

In [ ]:
TEK_DATA_DIR = "tektronik/"
#
DATABASE_DIR = os.path.join(DATA_DIR, TEK_DATA_DIR)
os.makedirs(DATABASE_DIR, exist_ok=True)
EXPERIMENT_NAME = "lock_acquisition"
EXPERIMENT_DIR = os.path.join(DATABASE_DIR, EXPERIMENT_NAME)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

print("Data will be saved in:", EXPERIMENT_DIR)

In [ ]:
# If True: save only channels that are currently displayed
# If False: try all listed channels regardless of display state
ONLY_DISPLAYED = True

# DPO2014B channels
CHANNELS = ["CH1", "CH2", "CH3", "CH4"]

In [22]:
# Low level USBTMC communication to acquire data from the scope
def write(fd, cmd: str) -> None:
    os.write(fd, (cmd + "\n").encode())


def read_line(fd) -> str:
    data = b""
    while True:
        c = os.read(fd, 1)
        if c == b"\n":
            break
        data += c
    return data.decode().strip()


def read_block(fd) -> bytes:
    """
    Read a SCPI definite-length binary block: #<N><LEN><DATA>
    """
    hdr = os.read(fd, 2)
    if len(hdr) < 2 or hdr[:1] != b"#":
        raise RuntimeError(f"Invalid binary block header: {hdr!r}")

    n_digits = int(hdr[1:2].decode())
    length = int(os.read(fd, n_digits).decode())

    data = b""
    while len(data) < length:
        chunk = os.read(fd, length - len(data))
        if not chunk:
            raise RuntimeError("Unexpected end of data while reading waveform block")
        data += chunk

    # Tek usually terminates with '\n' after the block
    # Try to consume it without failing if absent
    try:
        tail = os.read(fd, 1)
        if tail not in (b"", b"\n"):
            pass
    except OSError:
        pass

    return data

In [23]:
# Tektronik waveform helpers
def get_channel_display_state(fd, channel: str) -> bool:
    write(fd, f"SEL:{channel}?")
    ans = read_line(fd)
    return ans.strip() in ("1", "ON")


def setup_waveform_transfer(fd, channel: str) -> None:
    """
    Configure waveform transfer for one channel.
    """
    write(fd, f"DATA:SOURCE {channel}")
    write(fd, "DATA:ENC RIBINARY")   # signed integer binary
    write(fd, "DATA:WIDTH 2")        # int16
    write(fd, "DATA:START 1")
    write(fd, "DATA:STOP 10000000")  # effectively 'all available points'


def read_waveform_preamble(fd):
    """
    Read scaling parameters from WFMOutpre?.
    For Tek scopes this returns a semicolon-separated list.
    """
    write(fd, "WFMOutpre?")
    preamble = read_line(fd)
    fields = preamble.split(";")

    # Indexing follows the convention already used in your script
    xincr = float(fields[9])
    xzero = float(fields[10])
    ymult = float(fields[13])
    yzero = float(fields[14])
    yoff  = float(fields[15])

    return {
        "raw": preamble,
        "xincr": xincr,
        "xzero": xzero,
        "ymult": ymult,
        "yzero": yzero,
        "yoff": yoff,
    }


def read_waveform(fd, channel: str):
    """
    Read one channel waveform and return time axis, volts, raw ADC data, metadata.
    """
    setup_waveform_transfer(fd, channel)
    meta = read_waveform_preamble(fd)

    # Optional: get engineering units
    write(fd, "WFMOutpre:XUNit?")
    xunit = read_line(fd)

    write(fd, "WFMOutpre:YUNit?")
    yunit = read_line(fd)

    write(fd, "CURVE?")
    raw = read_block(fd)

    n_pts = len(raw) // 2
    adc = np.array(struct.unpack(f">{n_pts}h", raw), dtype=np.int16)

    volts = (adc - meta["yoff"]) * meta["ymult"] + meta["yzero"]
    time_axis = meta["xzero"] + np.arange(n_pts) * meta["xincr"]

    meta["xunit"] = xunit
    meta["yunit"] = yunit
    meta["n_pts"] = n_pts

    return time_axis, volts, adc, meta

In [24]:
# Quick connection test
fd = os.open(SCOPE_DEV, os.O_RDWR)
try:
    write(fd, "*IDN?")
    print(read_line(fd))
finally:
    os.close(fd)

TEKTRONIX,DPO2014B,C010359,CF:91.1CT FV:v1.52


In [ ]:
aqm = AcquisitionAnalysisManager(EXPERIMENT_DIR)
aqm.acquisition_cell(EXPERIMENT_NAME)

fd = os.open(SCOPE_DEV, os.O_RDWR)

try:
    write(fd, "*IDN?")
    idn = read_line(fd)
    print("Connected to:", idn)

    all_data = {
        "scope_idn": idn,
        "timestamp": time.time(),
        "channels_requested": CHANNELS,
    }

    saved_channels = []

    for ch in CHANNELS:
        try:
            displayed = get_channel_display_state(fd, ch)

            if ONLY_DISPLAYED and not displayed:
                print(f"{ch}: not displayed, skipping")
                continue

            print(f"{ch}: reading waveform...")
            t, v, adc, meta = read_waveform(fd, ch)

            all_data[f"{ch}_displayed"] = displayed
            all_data[f"{ch}_time"] = t
            all_data[f"{ch}_volts"] = v
            all_data[f"{ch}_adc"] = adc
            all_data[f"{ch}_meta"] = meta

            saved_channels.append(ch)
            print(f"{ch}: saved {len(v)} points")

        except Exception as e:
            print(f"{ch}: failed -> {e}")
            all_data[f"{ch}_error"] = str(e)

    all_data["saved_channels"] = saved_channels

    aqm.save_acquisition(**all_data)
    print("Saved acquisition with channels:", saved_channels)

finally:
    os.close(fd)

In [ ]:
# Quick figure plot
# Separated
for ch in all_data.get("saved_channels", []):
    t = all_data[f"{ch}_time"]
    v = all_data[f"{ch}_volts"]
    plt.figure()
    plt.plot(t, v)
    plt.title(f"{ch} - {all_data['scope_idn']}")
    plt.xlabel(all_data[f"{ch}_meta"]["xunit"])
    plt.ylabel(all_data[f"{ch}_meta"]["yunit"])
    plt.grid()

# Temperature sweep acquisition (passive oscilloscope readout)

This section:
- sets a temperature sweep on the OC3
- waits for stabilization
- asks for manual confirmation: `Signal Locked, proceed? [Y/n]`
- reads the trace from the chosen oscilloscope channel
- saves the data to a file (.h5 labmate format)
- plots mean intensity vs. temperature with error bars from the trace standard deviation

The oscilloscope is read in **passive mode**:
- no vertical scale change
- no horizontal scale change
- no acquisition mode change
- no point-number change

The only scope-side action is selecting the channel used for waveform transfer, then restoring the previous transfer source.

In [25]:
from oc_linux import OC3
import pandas as pd

# ============================================================
# Temperature sweep user parameters
# ============================================================

OC_PORT = "/dev/ttyUSB0"
TEMP_START = 42.5
TEMP_END = 43.5
TEMP_STEP = 0.2
RAMP_RATE = 0.05          # °C/s

TEMP_TOL = 0.01          # stabilization tolerance in °C
STABLE_TIME = 10.0       # seconds continuously within tolerance
POLL_INTERVAL = 1.0      # seconds between temperature checks

INTENSITY_CHANNEL = "CH3"  # Make sure to connect the power-meter to the set channel

TEMP_SCAN_DIR = os.path.join(DATA_DIR, "temperature_scan")
RUN_NAME = time.strftime("temp_scan_%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(TEMP_SCAN_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

print("Temperature scan data will be saved in:", RUN_DIR)

Temperature scan data will be saved in: tmp_data/temperature_scan/temp_scan_20260420_134228


In [26]:
aqm = AcquisitionAnalysisManager(RUN_DIR)
aqm.acquisition_cell(RUN_NAME)

INFO:1:2026_04_20__13_42_31__temp_scan_20260420_134228


In [27]:
def generate_temperature_list(start, end, step):
    if step <= 0:
        raise ValueError("TEMP_STEP must be > 0")

    temps = []
    if end >= start:
        t = start
        while t <= end + 1e-12:
            temps.append(round(t, 10))
            t += step
    else:
        t = start
        while t >= end - 1e-12:
            temps.append(round(t, 10))
            t -= step
    return temps


def get_oc3_temperature(oc):
    status = oc.get_status().decode()
    fields = status.split(";")
    return float(fields[1])


def wait_for_stable_temperature(oc, setpoint):
    """
    Wait until temperature is within TEMP_TOL for STABLE_TIME seconds.
    Returns the stabilized measured temperature.
    """
    t_stable = None

    while True:
        actual = get_oc3_temperature(oc)
        delta = abs(actual - setpoint)
        print(f"  T = {actual:.3f} °C | Δ = {delta:.3f} °C", end="\r")

        if delta < TEMP_TOL:
            if t_stable is None:
                t_stable = time.time()
            elif time.time() - t_stable >= STABLE_TIME:
                print()
                return actual
        else:
            t_stable = None

        time.sleep(POLL_INTERVAL)


def prompt_user_locked():
    while True:
        ans = input("Signal Locked, proceed? [Y/n] ").strip().lower()
        if ans in ("", "y", "yes"):
            return
        print("Waiting for lock confirmation...")

In [28]:
def read_waveform_passive(fd, channel: str):
    """
    Passive scope readout:
    - no acquisition-state command
    - no vertical/horizontal setting change
    - no acquisition mode change
    - no DATA:ENC / DATA:WIDTH / DATA:START / DATA:STOP change
    - only switch DATA:SOURCE to the requested channel for transfer
      and restore the previous source afterwards
    """
    write(fd, "DATA:SOURCE?")
    previous_source = read_line(fd).strip()

    try:
        if previous_source != channel:
            write(fd, f"DATA:SOURCE {channel}")

        meta = read_waveform_preamble(fd)

        write(fd, "WFMOutpre:XUNit?")
        xunit = read_line(fd)

        write(fd, "WFMOutpre:YUNit?")
        yunit = read_line(fd)

        write(fd, "CURVE?")
        raw = read_block(fd)

    finally:
        if previous_source and previous_source != channel:
            write(fd, f"DATA:SOURCE {previous_source}")

    n_pts = len(raw) // 2
    adc = np.array(struct.unpack(f">{n_pts}h", raw), dtype=np.int16)

    volts = (adc - meta["yoff"]) * meta["ymult"] + meta["yzero"]
    time_axis = meta["xzero"] + np.arange(n_pts) * meta["xincr"]

    meta["xunit"] = xunit
    meta["yunit"] = yunit
    meta["n_pts"] = n_pts
    meta["channel"] = channel
    meta["previous_source"] = previous_source

    return time_axis, volts, adc, meta


def save_temperature_trace_csv(filepath, t, v):
    arr = np.column_stack([t, v])
    np.savetxt(
        filepath,
        arr,
        delimiter=",",
        header="time_s,intensity_V",
        comments=""
    )

In [29]:
temps = generate_temperature_list(TEMP_START, TEMP_END, TEMP_STEP)

scan_results = []
oc = None
fd = None

try:
    oc = OC3(OC_PORT)
    time.sleep(0.3)
    oc.enable()

    fd = os.open(SCOPE_DEV, os.O_RDWR)
    time.sleep(0.2)

    write(fd, "*IDN?")
    scope_idn = read_line(fd)
    print("Connected to scope:", scope_idn)
    print("Temperature list:", temps)

    for i, T in enumerate(temps, start=1):
        print("\n" + "=" * 60)
        print(f"Point {i}/{len(temps)}  |  Set temperature = {T:.3f} °C")
        print("=" * 60)

        oc.set_temperature(T, ramp=RAMP_RATE)
        time.sleep(0.5)

        T_actual = wait_for_stable_temperature(oc, T)
        print(f"Temperature stabilized at {T_actual:.3f} °C")

        prompt_user_locked()

        print(f"Reading trace from {INTENSITY_CHANNEL} without changing scope settings...")
        t_axis, volts, adc, meta = read_waveform_passive(fd, INTENSITY_CHANNEL)

        mean_intensity = float(np.mean(volts))
        std_intensity = float(np.std(volts, ddof=1)) if len(volts) > 1 else 0.0

        trace_file = os.path.join(RUN_DIR, f"T_{T_actual:06.3f}C.csv".replace(".", "p", 1))
        save_temperature_trace_csv(trace_file, t_axis, volts)

        scan_results.append({
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "scope_idn": scope_idn,
            "channel": INTENSITY_CHANNEL,
            "temperature_setpoint_C": T,
            "temperature_measured_C": T_actual,
            "mean_intensity_V": mean_intensity,
            "std_intensity_V": std_intensity,
            "n_samples": len(volts),
            "trace_file": trace_file,
        })

        print(f"Saved trace: {trace_file}")
        print(f"Mean intensity = {mean_intensity:.6g} V")
        print(f"Std intensity  = {std_intensity:.6g} V")

finally:
    if fd is not None:
        os.close(fd)
    if oc is not None:
        try:
            oc.disable()
        except Exception:
            pass
        try:
            oc.close()
        except Exception:
            pass

summary_path = os.path.join(RUN_DIR, "temperature_scan_summary.csv")
df_scan = pd.DataFrame(scan_results)
df_scan.to_csv(summary_path, index=False)

print("\nSaved summary:", summary_path)
df_scan

Connected to scope: TEKTRONIX,DPO2014B,C010359,CF:91.1CT FV:v1.52
Temperature list: [42.5, 42.7, 42.9, 43.1, 43.3, 43.5]

Point 1/6  |  Set temperature = 42.500 °C
  T = 42.503 °C | Δ = 0.003 °C
Temperature stabilized at 42.503 °C
Reading trace from CH3 without changing scope settings...
Saved trace: tmp_data/temperature_scan/temp_scan_20260420_134228/T_42p503C.csv
Mean intensity = -0.00245013 V
Std intensity  = 0.00891287 V

Point 2/6  |  Set temperature = 42.700 °C
  T = 42.701 °C | Δ = 0.001 °C
Temperature stabilized at 42.701 °C
Reading trace from CH3 without changing scope settings...
Saved trace: tmp_data/temperature_scan/temp_scan_20260420_134228/T_42p701C.csv
Mean intensity = -0.000833007 V
Std intensity  = 0.0086985 V

Point 3/6  |  Set temperature = 42.900 °C
  T = 42.903 °C | Δ = 0.003 °C
Temperature stabilized at 42.903 °C
Reading trace from CH3 without changing scope settings...
Saved trace: tmp_data/temperature_scan/temp_scan_20260420_134228/T_42p903C.csv
Mean intensity =

TimeoutError: [Errno 110] Connection timed out

In [30]:
if len(df_scan) == 0:
    print("No points acquired.")
else:
    plt.figure(figsize=(8, 5))
    plt.errorbar(
        df_scan["temperature_measured_C"],
        df_scan["mean_intensity_V"],
        yerr=df_scan["std_intensity_V"],
        fmt="o-",
        capsize=4,
    )
    plt.xlabel("Temperature (°C)")
    plt.ylabel("Intensity (V)")
    plt.title(f"Intensity vs temperature ({INTENSITY_CHANNEL})")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

NameError: name 'df_scan' is not defined